<a href="https://colab.research.google.com/github/lovnishverma/Python-Getting-Started/blob/main/Module2_Pandas_Data_Manipulation_Aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 2: Data Manipulation and Aggregation using Pandas

A complete, beginner-friendly, exam-ready notebook. Every concept follows this pattern:

**Theory -> Syntax -> Example -> Actual Output -> Explanation -> Practical Example -> Common Mistakes -> Exam Tips -> Practice Question**



---

## Table of Contents

1. [Setup](#setup)
2. [Series and DataFrame](#s1)
3. [Creating and Exploring DataFrames](#s2)
4. [Indexing: loc vs iloc](#s3)
5. [Data Manipulation and Wrangling](#s4)
6. [Categorical and Numerical Data](#s5)
7. [Missing Values](#s6)
8. [Duplicate Values](#s7)
9. [Aggregation and GroupBy](#s8)
10. [apply(), map(), lambda, Vectorization](#s9)
11. [Pivot Tables and Crosstab](#s10)
12. [merge(), join(), concat()](#s11)
13. [Time-Series Analysis](#s12)
14. [Large-Dataset Optimization](#s13)
15. [Complete Data-Cleaning Workflow (Case Study)](#s14)
16. [Common Pandas Mistakes](#s15)
17. [Master Comparison Tables](#s16)
18. [Exam Revision and Cheat Sheet](#s17)
19. [Viva Questions (25, with answers)](#s18)
20. [Exam Question Bank](#s19)
21. [Hands-On Exercises](#s20)
22. [Solutions to Exercises](#s21)


<a id="setup"></a>
## 0. Setup

Run this cell first. It imports Pandas and NumPy and fixes random seeds so that every output in this notebook is reproducible (deterministic), as required for exam practice.

In [1]:
import pandas as pd
import numpy as np

# Fix random state everywhere so outputs are always identical
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print("Pandas version:", pd.__version__)
print("NumPy version :", np.__version__)


Pandas version: 2.2.3
NumPy version : 2.1.3


<a id="s1"></a>
## 1. Series and DataFrame

### Theory
Pandas has two core data structures:

- **Series**: a one-dimensional labeled array. It is like a single column of data with an index attached.
- **DataFrame**: a two-dimensional labeled table made of rows and columns. Each column of a DataFrame is internally a Series.

Both structures are built on top of NumPy arrays, which is why Pandas operations are fast and support vectorization.

### Syntax
```python
pd.Series(data, index=None, dtype=None, name=None)
pd.DataFrame(data, index=None, columns=None, dtype=None)
```

In [2]:
# Beginner Example: creating a Series
marks = pd.Series([56, 78, 90, 45], index=['Ravi', 'Simran', 'Aman', 'Neha'], name='Marks')
marks


,Marks
Ravi,56
Simran,78
Aman,90
Neha,45


**Explanation:** `marks` is a Series. The left column (`Ravi`, `Simran`, ...) is the *index* (labels), and the right column contains the *values*. The name `Marks` is attached as metadata.

In [3]:
# Beginner Example: creating a DataFrame from a dictionary
students = pd.DataFrame({
    'Name': ['Ravi', 'Simran', 'Aman', 'Neha'],
    'Marks': [56, 78, 90, 45],
    'City': ['Ludhiana', 'Delhi', 'Ludhiana', 'Patiala']
})
students


,Name,Marks,City
0,Ravi,56,Ludhiana
1,Simran,78,Delhi
2,Aman,90,Ludhiana
3,Neha,45,Patiala


**Explanation:** `students` is a DataFrame with three columns. Pandas automatically assigns a default integer index (0, 1, 2, 3).

### Practical Example
A single column pulled out of a DataFrame is a Series:

In [4]:
name_column = students['Name']
print(type(name_column))
name_column


<class 'pandas.core.series.Series'>


,Name
0,Ravi
1,Simran
2,Aman
3,Neha


### Series vs DataFrame — Comparison Table

| Feature | Series | DataFrame |
|---|---|---|
| Dimensions | 1-D | 2-D |
| Structure | Single column with index | Multiple columns (rows x columns) |
| Analogy | One column of Excel sheet | Entire Excel sheet |
| Creation | `pd.Series(data)` | `pd.DataFrame(data)` |
| Access element | `s[label]` or `s.iloc[pos]` | `df[col]`, `df.loc[row, col]` |
| Underlying type | NumPy array + Index | Dict of Series (columns) sharing an index |

### Common Mistakes
- Confusing `df['col']` (returns a Series) with `df[['col']]` (returns a DataFrame with one column).
- Forgetting that Series has both an index and values — treating it like a plain Python list.

### Exam Tips
- If asked "what is returned when you select one column", the safe exam answer is **Series** (unless double brackets are used).
- Remember: DataFrame = collection of Series sharing the same index.

### Practice Question
Create a Series of 5 city names with a custom index of your choice, then create a DataFrame with two columns: `City` and `Population` for the same 5 cities.

<a id="s2"></a>
## 2. Creating and Exploring DataFrames

### Theory
Before manipulating data, we must inspect it: its shape, column types, missing values, and summary statistics. Pandas offers dedicated inspection methods for this.

### Syntax
```python
df.head(n)      # first n rows
df.tail(n)      # last n rows
df.shape        # (rows, columns)
df.info()       # column dtypes + non-null counts
df.describe()   # summary statistics
df.columns      # column names
df.dtypes       # data type of each column
```

We will build one realistic dataset here and reuse it across most sections: a small **retail sales dataset** containing numeric, categorical, missing, duplicate, and date data (all created deterministically).

In [5]:
# Building the master dataset used throughout the notebook (deterministic)
rng = np.random.default_rng(42)

n = 20
dates = pd.date_range(start='2024-01-01', periods=n, freq='3D')
products = ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam']
regions = ['North', 'South', 'East', 'West']

sales = pd.DataFrame({
    'OrderID': range(1001, 1001 + n),
    'Date': dates,
    'Product': rng.choice(products, size=n),
    'Region': rng.choice(regions, size=n),
    'Quantity': rng.integers(1, 10, size=n),
    'Price': rng.choice([500, 800, 1200, 25000, 45000], size=n),
})
sales['Revenue'] = sales['Quantity'] * sales['Price']

# Deliberately inject missing values (deterministic positions)
sales.loc[[2, 7, 15], 'Quantity'] = np.nan
sales.loc[[5, 12], 'Region'] = np.nan

# Deliberately inject duplicate rows
sales = pd.concat([sales, sales.iloc[[3, 9]]], ignore_index=True)

sales


,OrderID,Date,Product,Region,Quantity,Price,Revenue
0,1001,2024-01-01,Laptop,East,2.0,45000,90000
1,1002,2024-01-04,Monitor,South,7.0,25000,175000
2,1003,2024-01-07,Monitor,North,NaN,800,5600
3,1004,2024-01-10,Keyboard,West,4.0,45000,180000
4,1005,2024-01-13,Keyboard,West,1.0,1200,1200
5,1006,2024-01-16,Webcam,NaN,9.0,800,7200
6,1007,2024-01-19,Laptop,South,5.0,45000,225000
7,1008,2024-01-22,Monitor,West,NaN,800,7200
8,1009,2024-01-25,Mouse,East,7.0,500,3500
9,1010,2024-01-28,Laptop,South,8.0,1200,9600


In [6]:
print("Shape:", sales.shape)
sales.info()


Shape: (22, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   OrderID   22 non-null     int64         
 1   Date      22 non-null     datetime64[ns]
 2   Product   22 non-null     object        
 3   Region    20 non-null     object        
 4   Quantity  19 non-null     float64       
 5   Price     22 non-null     int64         
 6   Revenue   22 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 1.3+ KB


In [7]:
sales.describe()


,OrderID,Date,Quantity,Price,Revenue
count,22.000000,22,19.000000,22.000000,22.000000
mean,1010.181818,2024-01-28 13:05:27.272727296,5.210526,13304.545455,63290.909091
min,1001.000000,2024-01-01 00:00:00,1.000000,500.000000,1000.000000
25%,1005.250000,2024-01-13 18:00:00,4.000000,800.000000,3625.000000
50%,1010.000000,2024-01-28 00:00:00,5.000000,1200.000000,7800.000000
75%,1014.750000,2024-02-11 06:00:00,7.000000,25000.000000,162500.000000
max,1020.000000,2024-02-27 00:00:00,9.000000,45000.000000,225000.000000
std,5.795206,NaN,2.347077,17886.346942,82621.523248


**Explanation:**
- `shape` returns `(rows, columns)`.
- `info()` shows each column's data type and how many non-null values it has — this is the fastest way to spot missing data.
- `describe()` gives count, mean, std, min, quartiles, and max, but **only for numeric columns by default**.

### Practical Example
To describe categorical (object) columns too, pass `include='object'`:

In [8]:
sales.describe(include='object')


,Product,Region
count,22,20
unique,5,4
top,Monitor,West
freq,7,7


### Common Mistakes
- Calling `describe()` and expecting statistics on text columns without passing `include='object'`.
- Reading `info()` output and ignoring the "non-null count" column, which is the key indicator of missing data.

### Exam Tips
- `df.shape` is an **attribute** (no parentheses); `df.info()` is a **method** (parentheses required).
- `describe()` by default excludes non-numeric columns.

### Practice Question
Using the `sales` DataFrame, print its column names, its data types, and the number of missing values per column (hint: use `isna().sum()`, covered later).

<a id="s3"></a>
## 3. Indexing: loc vs iloc

### Theory
Pandas provides two main indexers:
- **`loc`** — label-based indexing (uses row/column *names*).
- **`iloc`** — position-based indexing (uses integer *positions*, like a list).

### Syntax
```python
df.loc[row_label, column_label]
df.iloc[row_position, column_position]
```

In [9]:
# Beginner Example
print("loc example (label-based):")
print(sales.loc[0, 'Product'])   # row labeled 0, column 'Product'

print("\niloc example (position-based):")
print(sales.iloc[0, 2])          # first row, third column (position 2)


loc example (label-based):
Laptop

iloc example (position-based):
Laptop


In [10]:
# Slicing rows 2 to 5 and selected columns
print("Using loc (inclusive of end label):")
display(sales.loc[2:5, ['Product', 'Quantity']])

print("Using iloc (exclusive of end position, like Python slicing):")
display(sales.iloc[2:5, [2, 4]])


Using loc (inclusive of end label):


,Product,Quantity
2,Monitor,NaN
3,Keyboard,4.0
4,Keyboard,1.0
5,Webcam,9.0


Using iloc (exclusive of end position, like Python slicing):


,Product,Quantity
2,Monitor,NaN
3,Keyboard,4.0
4,Keyboard,1.0


**Explanation:** `loc[2:5]` includes row label 5, whereas `iloc[2:5]` excludes position 5 (standard Python slicing rule). This is one of the most common sources of off-by-one confusion.

### Practical Example
Filtering rows using `loc` with a boolean condition (very common in real work):

In [11]:
high_value_orders = sales.loc[sales['Revenue'] > 20000, ['OrderID', 'Product', 'Revenue']]
high_value_orders


,OrderID,Product,Revenue
0,1001,Laptop,90000
1,1002,Monitor,175000
3,1004,Keyboard,180000
6,1007,Laptop,225000
10,1011,Keyboard,175000
14,1015,Monitor,125000
19,1020,Keyboard,175000
20,1004,Keyboard,180000


### loc vs iloc — Comparison Table

| Feature | `loc` | `iloc` |
|---|---|---|
| Basis | Label / name based | Integer position based |
| Row 5 means | The row **labeled** 5 | The **6th** row (0-indexed) |
| Slice end | Inclusive | Exclusive |
| Boolean filtering | Yes, natural fit | Not directly (needs positions) |
| Column selection | By column name | By column integer position |

### Common Mistakes
- Using `iloc` with column names, or `loc` with integer positions when the index is not a simple range — this raises `KeyError` or `TypeError`.
- Forgetting that `loc` slicing is **inclusive** of the end label, unlike normal Python/`iloc` slicing.

### Exam Tips
- A favorite viva question: "What happens if you write `df.loc[0:3]` vs `df.iloc[0:3]`?" Answer: `loc` returns rows labeled 0,1,2,3 (4 rows); `iloc` returns positions 0,1,2 (3 rows).

### Practice Question
Using `sales`, retrieve the `Product` and `Price` of rows at *positions* 5 through 8 using `iloc`, and the same for rows *labeled* 5 through 8 using `loc`. Compare the results.

<a id="s4"></a>
## 4. Data Manipulation and Wrangling

### Theory
Data wrangling means transforming raw data into a clean, usable form: renaming columns, adding/removing columns, sorting, filtering, and changing values.

### Syntax
```python
df.rename(columns={'old': 'new'})
df.drop(columns=['col'])
df.sort_values(by='col', ascending=True)
df['new_col'] = ...
df.assign(new_col=...)
```

In [12]:
# Beginner Example: renaming a column
sales_renamed = sales.rename(columns={'OrderID': 'Order_ID'})
sales_renamed.columns.tolist()


['Order_ID', 'Date', 'Product', 'Region', 'Quantity', 'Price', 'Revenue']

In [13]:
# Adding a new column
sales['HighValue'] = sales['Revenue'] > 20000
sales.head()


,OrderID,Date,Product,Region,Quantity,Price,Revenue,HighValue
0,1001,2024-01-01,Laptop,East,2.0,45000,90000,True
1,1002,2024-01-04,Monitor,South,7.0,25000,175000,True
2,1003,2024-01-07,Monitor,North,NaN,800,5600,False
3,1004,2024-01-10,Keyboard,West,4.0,45000,180000,True
4,1005,2024-01-13,Keyboard,West,1.0,1200,1200,False


**Explanation:** `HighValue` is a new boolean column created by applying a condition directly on the `Revenue` column (vectorized comparison — no loop needed).

In [14]:
# Sorting values
sales.sort_values(by='Revenue', ascending=False).head(5)


,OrderID,Date,Product,Region,Quantity,Price,Revenue,HighValue
6,1007,2024-01-19,Laptop,South,5.0,45000,225000,True
3,1004,2024-01-10,Keyboard,West,4.0,45000,180000,True
20,1004,2024-01-10,Keyboard,West,4.0,45000,180000,True
19,1020,2024-02-27,Keyboard,East,7.0,25000,175000,True
10,1011,2024-01-31,Keyboard,South,7.0,25000,175000,True


In [15]:
# Dropping a column (does not modify original unless inplace=True)
sales_dropped = sales.drop(columns=['HighValue'])
sales_dropped.columns.tolist()


['OrderID', 'Date', 'Product', 'Region', 'Quantity', 'Price', 'Revenue']

### Practical Example
Filtering with multiple conditions using `&` (and) / `|` (or) — each condition **must** be wrapped in parentheses:

In [16]:
filtered = sales[(sales['Region'] == 'North') & (sales['Quantity'] > 3)]
filtered


,OrderID,Date,Product,Region,Quantity,Price,Revenue,HighValue


### Common Mistakes
- Using Python's `and`/`or` instead of `&`/`|` for combining boolean Series — this raises a `ValueError`.
- Forgetting parentheses around each condition when combining them, e.g. `sales['Region']=='North' & sales['Quantity']>3` is evaluated incorrectly due to operator precedence.
- Assuming operations like `drop()` or `sort_values()` modify the DataFrame in place by default — they return a **new** DataFrame unless `inplace=True` is passed.

### Exam Tips
- `inplace=True` returns `None`; a very common exam trick question is: `df2 = df.drop(columns=['x'], inplace=True)` then `df2` is `None`.
- Sorting: `ascending=False` for descending order.

### Practice Question
From `sales`, create a new column `RevenuePerUnit = Revenue / Quantity`, then sort the DataFrame by this new column in descending order and show the top 3 rows.

<a id="s5"></a>
## 5. Categorical and Numerical Data

### Theory
- **Numerical data**: quantities that can be measured (e.g., `Price`, `Quantity`). Stored as `int64` / `float64`.
- **Categorical data**: data that takes a limited, fixed set of values (e.g., `Region`, `Product`). Stored as `object` (string) by default, but converting to Pandas' `category` dtype saves memory and speeds up group operations.

### Syntax
```python
df['col'] = df['col'].astype('category')
df['col'].cat.categories
df['col'].value_counts()
```

In [17]:
# Beginner Example: checking current dtype
print(sales['Region'].dtype)
sales['Region'].value_counts(dropna=False)


object


,count
Region,
West,7
South,6
East,4
North,3
NaN,2


In [18]:
# Converting to category dtype
sales['Product'] = sales['Product'].astype('category')
print(sales['Product'].dtype)
sales['Product'].cat.categories


category


Index(['Keyboard', 'Laptop', 'Monitor', 'Mouse', 'Webcam'], dtype='object')

**Explanation:** `value_counts()` counts how many times each unique value appears — the standard first step to understand a categorical column. `astype('category')` converts the column to Pandas' memory-efficient categorical type.

### Practical Example
Categorical data can be ordered (ordinal), which matters for sorting and comparisons:

In [19]:
priority = pd.Series(['Low', 'High', 'Medium', 'Low', 'High'])
priority_cat = pd.Categorical(priority, categories=['Low', 'Medium', 'High'], ordered=True)
priority_cat.sort_values()


['Low', 'Low', 'Medium', 'High', 'High']
Categories (3, object): ['Low' < 'Medium' < 'High']

### Common Mistakes
- Treating categorical text columns as numeric and trying to compute `mean()` on them (raises `TypeError`).
- Forgetting `ordered=True` when the categories have a natural rank (Low < Medium < High), which prevents correct comparison operators (`<`, `>`).

### Exam Tips
- `category` dtype uses less memory than `object` for columns with few unique repeated values — a common "why use category dtype" answer.
- `value_counts(normalize=True)` gives proportions instead of counts.

### Practice Question
Convert the `Region` column to category dtype, then use `value_counts(normalize=True)` to find what percentage of orders came from each region.

<a id="s6"></a>
## 6. Missing Values

### Theory
Real-world data almost always has missing values, represented in Pandas as `NaN` (Not a Number). Handling them correctly is essential before any analysis.

### Syntax
```python
df.isna()          # boolean mask of missing values
df.isna().sum()     # count of missing values per column
df.dropna()         # remove rows/columns with missing values
df.fillna(value)    # fill missing values
df['col'].fillna(df['col'].mean())
```

In [20]:
# Beginner Example
print("Missing values per column:")
sales.isna().sum()


Missing values per column:


,0
OrderID,0
Date,0
Product,0
Region,2
Quantity,3
Price,0
Revenue,0
HighValue,0


In [21]:
# dropna: removes rows with ANY missing value
print("Original rows:", len(sales))
dropped = sales.dropna()
print("Rows after dropna():", len(dropped))


Original rows: 22
Rows after dropna(): 17


In [22]:
# fillna: filling numeric column with the mean, categorical with 'Unknown'
sales_filled = sales.copy()
sales_filled['Quantity'] = sales_filled['Quantity'].fillna(sales_filled['Quantity'].mean())
sales_filled['Region'] = sales_filled['Region'].fillna('Unknown')
sales_filled.isna().sum()


,0
OrderID,0
Date,0
Product,0
Region,0
Quantity,0
Price,0
Revenue,0
HighValue,0


**Explanation:** `dropna()` deletes entire rows containing at least one `NaN`, which can discard useful data. `fillna()` is usually preferred because it preserves rows, using the column mean (numeric) or a placeholder label (categorical).

### Practical Example
Forward-fill and backward-fill are useful for time-ordered data:

In [23]:
demo = pd.Series([10, np.nan, np.nan, 40, np.nan])
print("Original:      ", demo.tolist())
print("ffill (forward):", demo.ffill().tolist())
print("bfill (backward):", demo.bfill().tolist())


Original:       [10.0, nan, nan, 40.0, nan]
ffill (forward): [10.0, 10.0, 10.0, 40.0, 40.0]
bfill (backward): [10.0, 40.0, 40.0, 40.0, nan]


### Common Mistakes
- Using `dropna()` on the whole DataFrame without checking how many rows will be lost — this can silently remove a large portion of the dataset.
- Filling missing numeric values with `0` when `0` is not a meaningful default (it can distort the mean/sum).
- Forgetting `dropna(subset=[...])` when you only want to drop rows missing values in specific columns.

### Exam Tips
- `df.isna()` and `df.isnull()` are exact aliases — either is acceptable in exams.
- `dropna(how='all')` drops a row only if **all** values are missing; default `how='any'` drops if **any** value is missing.

### Practice Question
Count the missing values in `sales` per column, then fill missing `Quantity` values with the column median instead of the mean, and compare the two results.

<a id="s7"></a>
## 7. Duplicate Values

### Theory
Duplicate rows can occur due to data entry errors or repeated imports and must be detected and removed for accurate analysis.

### Syntax
```python
df.duplicated()             # boolean Series, True for duplicate rows
df.duplicated(subset=[...]) # check duplicates on specific columns
df.drop_duplicates()        # remove duplicate rows
```

In [24]:
# Beginner Example
print("Number of duplicate rows:", sales.duplicated().sum())
sales[sales.duplicated()]


Number of duplicate rows: 2


,OrderID,Date,Product,Region,Quantity,Price,Revenue,HighValue
20,1004,2024-01-10,Keyboard,West,4.0,45000,180000,True
21,1010,2024-01-28,Laptop,South,8.0,1200,9600,False


In [25]:
# Removing duplicates
sales_unique = sales.drop_duplicates()
print("Rows before:", len(sales), " Rows after:", len(sales_unique))


Rows before: 22  Rows after: 20


**Explanation:** `duplicated()` marks the **second and later** occurrences of an identical row as `True` by default (the first occurrence is kept as `False`). `drop_duplicates()` removes those marked rows.

### Practical Example
Checking duplicates based on a subset of columns (e.g., same `OrderID` should never repeat):

In [26]:
print(sales.duplicated(subset=['OrderID']).sum(), "duplicate OrderIDs found")


2 duplicate OrderIDs found


### Common Mistakes
- Assuming `drop_duplicates()` checks all columns when you actually need `subset=[...]` for a business key like `OrderID`.
- Not deciding which occurrence to keep — `keep='first'` (default), `keep='last'`, or `keep=False` (drop all copies).

### Exam Tips
- `duplicated(keep=False)` marks **all** copies (including the first) as duplicates — useful to inspect every duplicate row together.

### Practice Question
Find and display all duplicate rows in `sales` based on the combination of `Product` and `Date`, keeping all occurrences visible.

<a id="s8"></a>
## 8. Aggregation and GroupBy

### Theory
`groupby()` implements the **split-apply-combine** pattern: split data into groups based on a key, apply an aggregation function to each group, then combine results into a new DataFrame/Series.

### Syntax
```python
df.groupby('col')['target'].sum()
df.groupby(['col1', 'col2']).agg({'target': ['sum', 'mean']})
```

In [27]:
# Use the filled dataset (no NaN) for clean aggregation examples
clean_sales = sales_filled.drop_duplicates()

# Beginner Example: total revenue per region
clean_sales.groupby('Region')['Revenue'].sum()


,Revenue
Region,
East,271000
North,7800
South,593000
Unknown,12000
West,319000


In [28]:
# Multiple aggregations at once
clean_sales.groupby('Product', observed=True)['Revenue'].agg(['sum', 'mean', 'count'])


,sum,mean,count
Product,,,
Keyboard,535200,107040.000000,5
Laptop,326200,81550.000000,4
Monitor,321300,45900.000000,7
Mouse,3500,3500.000000,1
Webcam,16600,5533.333333,3


**Explanation:** `groupby('Region')` splits data into one group per unique region; `['Revenue'].sum()` applies the sum function to each group; the result combines all group sums into a single Series indexed by `Region`. `.agg([...])` allows several statistics at once.

### Practical Example
Grouping by multiple columns:

In [29]:
clean_sales.groupby(['Region', 'Product'], observed=True)['Revenue'].sum().head(10)


Region  Product 
East    Keyboard    175000
        Laptop       90000
        Monitor       2500
        Mouse         3500
North   Monitor       6800
        Webcam        1000
South   Keyboard    175000
        Laptop      234600
        Monitor     175000
        Webcam        8400
Name: Revenue, dtype: int64

### count vs size — Comparison Table

| Feature | `count()` | `size()` |
|---|---|---|
| Counts | Non-null values only, per column | All rows in the group (including NaN) |
| Return type on groupby | DataFrame (one count per column) | Series (single number per group) |
| Use case | Checking missing data per column within a group | Simply counting group membership |

### mean vs median — Comparison Table

| Feature | `mean()` | `median()` |
|---|---|---|
| Definition | Arithmetic average | Middle value when sorted |
| Sensitivity to outliers | High | Low (robust) |
| Best used when | Data is roughly symmetric | Data is skewed / has outliers |

In [30]:
print("count per group:\n", clean_sales.groupby('Region', observed=True)['Revenue'].count())
print("\nsize per group:\n", clean_sales.groupby('Region', observed=True).size())


count per group:
 Region
East       4
North      3
South      5
Unknown    2
West       6
Name: Revenue, dtype: int64

size per group:
 Region
East       4
North      3
South      5
Unknown    2
West       6
dtype: int64


### Common Mistakes
- Calling `.groupby('col').mean()` on a DataFrame that still has non-numeric columns without selecting numeric columns first (older Pandas errors; current Pandas silently drops them but it's best practice to select explicitly).
- Forgetting that `groupby()` alone returns a `GroupBy` object, not a DataFrame — an aggregation function must be applied.
- Using `count()` when you actually meant `size()` (they differ when NaNs are present).

### Exam Tips
- The standard exam phrase "split-apply-combine" describes exactly how `groupby` works — always mention it in theory answers.
- `reset_index()` after a groupby aggregation converts the grouped index back into a normal column, often required before further processing.

### Practice Question
Group `clean_sales` by `Region` and calculate the mean, minimum, and maximum `Quantity` sold, in a single `agg()` call.

<a id="s9"></a>
## 9. apply(), map(), lambda, and Vectorization

### Theory
- **`map()`**: element-wise transformation, works only on a **Series**. Good for substituting/mapping values.
- **`apply()`**: works on Series (element-wise) or DataFrame (row-wise/column-wise via `axis`). More general, can call any function.
- **`lambda`**: an anonymous, throwaway function, often used inside `apply`/`map`.
- **Vectorization**: applying an operation to an entire array/Series at once (using Pandas/NumPy's internal optimized C code) instead of looping — this is far faster than `apply`.

### Syntax
```python
series.map(func_or_dict)
df['col'].apply(func)
df.apply(func, axis=1)   # row-wise
df['col'] * 2            # vectorized operation
```

In [31]:
# Beginner Example: map with a dictionary
region_code = {'North': 'N', 'South': 'S', 'East': 'E', 'West': 'W'}
clean_sales['RegionCode'] = clean_sales['Region'].map(region_code)
clean_sales[['Region', 'RegionCode']].head()


/tmp/ipykernel_729/1336754596.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_sales['RegionCode'] = clean_sales['Region'].map(region_code)


,Region,RegionCode
0,East,E
1,South,S
2,North,N
3,West,W
4,West,W


In [32]:
# apply with lambda on a Series
clean_sales['PriceCategory'] = clean_sales['Price'].apply(lambda p: 'Expensive' if p > 5000 else 'Affordable')
clean_sales[['Price', 'PriceCategory']].head()


/tmp/ipykernel_729/1532494030.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_sales['PriceCategory'] = clean_sales['Price'].apply(lambda p: 'Expensive' if p > 5000 else 'Affordable')


,Price,PriceCategory
0,45000,Expensive
1,25000,Expensive
2,800,Affordable
3,45000,Expensive
4,1200,Affordable


In [33]:
# apply row-wise (axis=1) on a DataFrame
clean_sales['Summary'] = clean_sales.apply(lambda row: f"{row['Product']} x{int(row['Quantity'])}", axis=1)
clean_sales[['Product', 'Quantity', 'Summary']].head()


/tmp/ipykernel_729/472056507.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_sales['Summary'] = clean_sales.apply(lambda row: f"{row['Product']} x{int(row['Quantity'])}", axis=1)


,Product,Quantity,Summary
0,Laptop,2.000000,Laptop x2
1,Monitor,7.000000,Monitor x7
2,Monitor,5.210526,Monitor x5
3,Keyboard,4.000000,Keyboard x4
4,Keyboard,1.000000,Keyboard x1


**Explanation:** `axis=1` tells `apply` to pass each **row** (as a Series) to the function; `axis=0` (default) would pass each **column**.

### Practical Example: Vectorization vs apply (performance)
Vectorized operations are almost always preferred over `apply` for numeric computations because they avoid Python-level looping.

In [34]:
import time

big = pd.DataFrame({'x': np.arange(200_000)})

start = time.time()
big['y_apply'] = big['x'].apply(lambda v: v * 2)
t_apply = time.time() - start

start = time.time()
big['y_vectorized'] = big['x'] * 2
t_vector = time.time() - start

print(f"apply time      : {t_apply:.4f} sec")
print(f"vectorized time : {t_vector:.4f} sec")
print("Vectorization is faster by a factor of approximately", round(t_apply / max(t_vector, 1e-9), 1))


apply time      : 0.0951 sec
vectorized time : 0.0029 sec
Vectorization is faster by a factor of approximately 33.3


### Common Mistakes
- Using `apply()` for simple arithmetic that could be vectorized directly (e.g., `df['col'].apply(lambda x: x*2)` instead of `df['col']*2`) — this is much slower on large data.
- Trying to use `.map()` on a whole DataFrame — `map()` only works on a Series (use `applymap`/`df.apply` equivalents for a DataFrame; `DataFrame.map` exists in modern Pandas as the elementwise version).
- Forgetting `axis=1` when a row-wise (not column-wise) operation is intended.

### Exam Tips
- Golden rule for exams: "Vectorized operations are faster than `apply`, which is faster than a manual Python `for` loop."
- `map()` = Series only, element-wise. `apply()` = Series or DataFrame, more flexible.

### Practice Question
Use `apply()` with a lambda to create a new column `RevenueTier` that labels each row as `'High'` if `Revenue > 20000`, else `'Low'`. Then rewrite the same logic using a fully vectorized approach with `np.where`.

<a id="s10"></a>
## 10. Pivot Tables and Crosstab

### Theory
- **`pivot_table()`**: reshapes data by turning unique values of one column into new columns, aggregating a numeric target — a more flexible, aggregation-aware version of `groupby`.
- **`crosstab()`**: computes a frequency table (cross-tabulation) between two or more categorical columns, by default counting occurrences.

### Syntax
```python
pd.pivot_table(df, values='target', index='row_col', columns='col_col', aggfunc='sum')
pd.crosstab(df['col1'], df['col2'])
```

In [35]:
# Beginner Example: pivot_table - total revenue by Region (rows) and Product (columns)
pivot = pd.pivot_table(clean_sales, values='Revenue', index='Region', columns='Product',
                        aggfunc='sum', fill_value=0, observed=True)
pivot


Product,Keyboard,Laptop,Monitor,Mouse,Webcam
Region,,,,,
East,175000,90000,2500,3500,0
North,0,0,6800,0,1000
South,175000,234600,175000,0,8400
Unknown,0,0,4800,0,7200
West,185200,1600,132200,0,0


In [36]:
# crosstab - count of orders by Region and Product
ct = pd.crosstab(clean_sales['Region'], clean_sales['Product'])
ct


Product,Keyboard,Laptop,Monitor,Mouse,Webcam
Region,,,,,
East,1,1,1,1,0
North,0,0,2,0,1
South,1,2,1,0,1
Unknown,0,0,1,0,1
West,3,1,2,0,0


**Explanation:** `pivot_table` summarizes a **numeric** value (`Revenue`) using an aggregation function (`sum`). `crosstab` simply **counts** how many rows fall into each combination of categories by default (though it also supports `values` + `aggfunc` for other aggregations).

### groupby vs pivot_table — Comparison Table

| Feature | `groupby()` | `pivot_table()` |
|---|---|---|
| Output shape | Long format (one row per group) | Wide format (categories become columns) |
| Readability for 2 keys | Multi-index, harder to read | Grid/table layout, easy to read |
| Default aggregation | Must specify explicitly | `mean` by default |
| Missing combinations | Simply absent | Can be filled with `fill_value` |
| Typical use | General-purpose aggregation | Cross-tabular summary reports |

### Practical Example
`pivot_table` with multiple aggregation functions:

In [37]:
pd.pivot_table(clean_sales, values='Revenue', index='Region', aggfunc=['sum', 'mean', 'count'], observed=True)


,sum,mean,count
,Revenue,Revenue,Revenue
Region,,,
East,271000,67750.000000,4
North,7800,2600.000000,3
South,593000,118600.000000,5
Unknown,12000,6000.000000,2
West,319000,53166.666667,6


### Common Mistakes
- Confusing `pivot()` (no aggregation, fails on duplicate index/column pairs) with `pivot_table()` (aggregates automatically, handles duplicates).
- Forgetting `fill_value=0` in `pivot_table`, leaving `NaN` for missing category combinations, which can break later arithmetic.
- Using `crosstab` when a numeric aggregation (not just counting) was actually needed — remember `crosstab` also accepts `values=` and `aggfunc=` for that case.

### Exam Tips
- Definition to memorize: "`pivot_table` aggregates numeric data into a grid; `crosstab` counts frequencies between categorical variables."

### Practice Question
Create a pivot table showing the **average** `Quantity` sold for each combination of `Region` and `Product`, filling missing combinations with 0.

<a id="s11"></a>
## 11. merge(), join(), and concat()

### Theory
- **`concat()`**: stacks DataFrames together, either vertically (more rows) or horizontally (more columns). Does not align on keys, just on index/axis.
- **`merge()`**: combines DataFrames based on common column(s) (like SQL joins), matching rows where key values are equal.
- **`join()`**: a convenience method for combining DataFrames primarily using their **index** (similar to a simplified `merge`).

### Syntax
```python
pd.concat([df1, df2], axis=0)          # stack rows
pd.concat([df1, df2], axis=1)          # stack columns
pd.merge(df1, df2, on='key', how='inner')
df1.join(df2, how='left')
```

In [38]:
# Sample DataFrames for merge/join/concat demonstrations
products_info = pd.DataFrame({
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Tablet'],
    'Category': ['Electronics', 'Accessory', 'Accessory', 'Electronics', 'Electronics']
})

region_manager = pd.DataFrame({
    'Region': ['North', 'South', 'East'],
    'Manager': ['Aarav', 'Divya', 'Karan']
})

print("products_info:")
display(products_info)
print("region_manager:")
display(region_manager)


products_info:


,Product,Category
0,Laptop,Electronics
1,Mouse,Accessory
2,Keyboard,Accessory
3,Monitor,Electronics
4,Tablet,Electronics


region_manager:


,Region,Manager
0,North,Aarav
1,South,Divya
2,East,Karan


In [39]:
# concat: stacking two DataFrames with the SAME columns, vertically
batch1 = clean_sales.iloc[:3][['Product', 'Region', 'Revenue']]
batch2 = clean_sales.iloc[3:6][['Product', 'Region', 'Revenue']]
combined = pd.concat([batch1, batch2], axis=0, ignore_index=True)
combined


,Product,Region,Revenue
0,Laptop,East,90000
1,Monitor,South,175000
2,Monitor,North,5600
3,Keyboard,West,180000
4,Keyboard,West,1200
5,Webcam,Unknown,7200


In [40]:
# merge: inner join (only matching Products kept)
merged_inner = pd.merge(clean_sales, products_info, on='Product', how='inner')
merged_inner[['Product', 'Category', 'Revenue']].head()


,Product,Category,Revenue
0,Laptop,Electronics,90000
1,Monitor,Electronics,175000
2,Monitor,Electronics,5600
3,Keyboard,Accessory,180000
4,Keyboard,Accessory,1200


In [41]:
# merge: left join (keep ALL rows from clean_sales, Region without a manager becomes NaN)
merged_left = pd.merge(clean_sales, region_manager, on='Region', how='left')
merged_left[['Region', 'Manager']].drop_duplicates()


,Region,Manager
0,East,Karan
1,South,Divya
2,North,Aarav
3,West,NaN
5,Unknown,NaN


**Explanation:** With `how='inner'`, only rows whose `Product` exists in **both** DataFrames survive (`Tablet` and `Webcam` disappear since they don't exist in both). With `how='left'`, every row of the left DataFrame (`clean_sales`) is kept, and `Manager` is `NaN` wherever no matching `Region` exists (e.g., `West` has no manager listed).

### Practical Example: join() using index

In [42]:
left_idx = clean_sales.set_index('Region')[['Product']].head(5)
right_idx = region_manager.set_index('Region')
left_idx.join(right_idx, how='left')


,Product,Manager
Region,,
East,Laptop,Karan
South,Monitor,Divya
North,Monitor,Aarav
West,Keyboard,NaN
West,Keyboard,NaN


### merge vs concat vs join — Comparison Table

| Feature | `merge()` | `concat()` | `join()` |
|---|---|---|---|
| Combines on | Common column(s) (keys) | Position / axis (index alignment) | Index (mainly) |
| Like SQL | JOIN | UNION (axis=0) | JOIN (index-based) |
| Adds | New columns matched by key | More rows or more columns | New columns matched by index |
| Flexibility | Most flexible (many `how` options) | Simplest, fastest for stacking | Convenience wrapper around merge |

### Inner / Left / Right / Outer Joins — Comparison Table

| Join Type | Rows Kept |
|---|---|
| `inner` | Only rows with matching keys in **both** DataFrames |
| `left` | All rows from the **left** DataFrame; unmatched right columns become `NaN` |
| `right` | All rows from the **right** DataFrame; unmatched left columns become `NaN` |
| `outer` | All rows from **both** DataFrames; unmatched cells become `NaN` |

In [43]:
# Demonstrating all four join types with small, clear tables
left_df = pd.DataFrame({'key': ['A', 'B', 'C'], 'left_val': [1, 2, 3]})
right_df = pd.DataFrame({'key': ['B', 'C', 'D'], 'right_val': [20, 30, 40]})

for how in ['inner', 'left', 'right', 'outer']:
    print(f"--- how='{how}' ---")
    display(pd.merge(left_df, right_df, on='key', how=how))


--- how='inner' ---


,key,left_val,right_val
0,B,2,20
1,C,3,30


--- how='left' ---


,key,left_val,right_val
0,A,1,NaN
1,B,2,20.0
2,C,3,30.0


--- how='right' ---


,key,left_val,right_val
0,B,2.0,20
1,C,3.0,30
2,D,NaN,40


--- how='outer' ---


,key,left_val,right_val
0,A,1.0,NaN
1,B,2.0,20.0
2,C,3.0,30.0
3,D,NaN,40.0


### axis=0 vs axis=1 — Comparison Table

| | `axis=0` | `axis=1` |
|---|---|---|
| Direction | Down the rows (row-wise operation, per column) | Across the columns (column-wise operation, per row) |
| `concat` meaning | Stack DataFrames vertically (more rows) | Stack DataFrames horizontally (more columns) |
| `drop` meaning | Drop rows | Drop columns |
| `apply` meaning | Apply function to each column | Apply function to each row |
| Mnemonic | 0 = top-to-bottom | 1 = left-to-right |

### Common Mistakes
- Using `concat()` when a key-based `merge()` was actually needed — `concat` does not match on values, only stacks blindly.
- Forgetting `on='key'` in `merge()` when column names differ between the two DataFrames (use `left_on=`/`right_on=` instead).
- Not checking for duplicate keys before merging, which can silently multiply rows (a many-to-many merge).

### Exam Tips
- Memorize the join table above — it is one of the most frequently asked theory + practical questions.
- `pd.merge(..., indicator=True)` adds a `_merge` column showing whether each row came from `left_only`, `right_only`, or `both` — useful for debugging joins.

### Practice Question
Merge `clean_sales` with `products_info` using an **outer** join on `Product`, and identify which products appear in only one of the two DataFrames using `indicator=True`.

<a id="s12"></a>
## 12. Time-Series Analysis

### Theory
Pandas has first-class support for dates and times through the `datetime64` dtype, `DatetimeIndex`, and resampling tools — essential for sales trends, stock prices, sensor logs, etc.

### Syntax
```python
pd.to_datetime(df['col'])
df.set_index('date_col')
df.resample('M').sum()      # resample by month
df['col'].dt.year / .dt.month / .dt.day_name()
df['col'].rolling(window=3).mean()
```

In [44]:
# Beginner Example: confirming Date column is datetime type
print(clean_sales['Date'].dtype)
clean_sales[['Date']].head()


datetime64[ns]


,Date
0,2024-01-01
1,2024-01-04
2,2024-01-07
3,2024-01-10
4,2024-01-13


In [45]:
# Extracting date parts
clean_sales['Year'] = clean_sales['Date'].dt.year
clean_sales['Month'] = clean_sales['Date'].dt.month
clean_sales['DayName'] = clean_sales['Date'].dt.day_name()
clean_sales[['Date', 'Year', 'Month', 'DayName']].head()


/tmp/ipykernel_729/2831860840.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_sales['Year'] = clean_sales['Date'].dt.year
/tmp/ipykernel_729/2831860840.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_sales['Month'] = clean_sales['Date'].dt.month
/tmp/ipykernel_729/2831860840.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.o

,Date,Year,Month,DayName
0,2024-01-01,2024,1,Monday
1,2024-01-04,2024,1,Thursday
2,2024-01-07,2024,1,Sunday
3,2024-01-10,2024,1,Wednesday
4,2024-01-13,2024,1,Saturday


**Explanation:** The `.dt` accessor exposes date/time components of a datetime column, similar to how `.str` exposes string methods.

### Practical Example: Resampling
Resampling groups time-series data into fixed time buckets (e.g., weekly totals):

In [46]:
ts = clean_sales.set_index('Date').sort_index()
weekly_revenue = ts['Revenue'].resample('W').sum()
weekly_revenue


,Revenue
Date,
2024-01-07,270600
2024-01-14,181200
2024-01-21,232200
2024-01-28,20300
2024-02-04,176000
2024-02-11,7300
2024-02-18,130200
2024-02-25,10000
2024-03-03,175000


In [47]:
# Rolling average (moving average) - smooths short-term fluctuations
ts_sorted = ts.sort_index()
ts_sorted['Revenue_RollingMean3'] = ts_sorted['Revenue'].rolling(window=3).mean()
ts_sorted[['Revenue', 'Revenue_RollingMean3']].head(8)


,Revenue,Revenue_RollingMean3
Date,,
2024-01-01,90000,NaN
2024-01-04,175000,NaN
2024-01-07,5600,90200.000000
2024-01-10,180000,120200.000000
2024-01-13,1200,62266.666667
2024-01-16,7200,62800.000000
2024-01-19,225000,77800.000000
2024-01-22,7200,79800.000000


**Explanation:** `resample('W')` groups all rows into weekly bins (like `groupby` but for time). `rolling(window=3).mean()` computes a moving average over the last 3 rows — the first two values are `NaN` because there aren't yet 3 prior rows to average.

### Common Mistakes
- Forgetting to convert a date column with `pd.to_datetime()` before using `.dt` or `resample()` — string dates do not support these operations.
- Not sorting by date before applying `rolling()` or `resample()`, producing misleading results.
- Confusing `resample()` (regular time buckets) with `groupby()` (arbitrary categorical grouping) — `resample` requires a `DatetimeIndex`.

### Exam Tips
- Common frequency codes: `'D'` = day, `'W'` = week, `'M'` = month end, `'Q'` = quarter, `'Y'` = year.
- `rolling(window=n)` always produces `n-1` leading `NaN` values.

### Practice Question
Using `ts_sorted`, compute the monthly total revenue using `resample('ME').sum()` and identify which month had the highest revenue.

<a id="s13"></a>
## 13. Large-Dataset Optimization

### Theory
As datasets grow, memory and speed become critical. Key optimization techniques:
1. Use the smallest sufficient numeric dtype (`int8`, `int32` instead of default `int64`).
2. Convert repeated text columns to `category` dtype.
3. Read only needed columns (`usecols`) and process files in `chunksize` batches for very large CSV files.
4. Prefer vectorized operations over `apply()`/loops.
5. Use `df.memory_usage(deep=True)` to measure actual memory used.

### Syntax
```python
df.memory_usage(deep=True)
df['col'] = df['col'].astype('int32')
pd.read_csv('file.csv', usecols=[...], dtype={...}, chunksize=100000)
```

In [48]:
# Beginner Example: memory usage before and after optimization
demo_df = pd.DataFrame({
    'id': np.arange(100_000),
    'category': np.random.choice(['A', 'B', 'C'], size=100_000),
    'value': np.random.randint(0, 100, size=100_000)
})

before = demo_df.memory_usage(deep=True).sum()

optimized = demo_df.copy()
optimized['id'] = optimized['id'].astype('int32')
optimized['category'] = optimized['category'].astype('category')
optimized['value'] = optimized['value'].astype('int8')

after = optimized.memory_usage(deep=True).sum()

print(f"Memory before optimization: {before/1024:.1f} KB")
print(f"Memory after optimization : {after/1024:.1f} KB")
print(f"Reduction: {(1 - after/before)*100:.1f}%")


Memory before optimization: 6445.4 KB
Memory after optimization : 586.3 KB
Reduction: 90.9%


**Explanation:** Converting a repeated-text column to `category` and using smaller integer types drastically reduces memory because Pandas stores categories once and references them by small integer codes, and smaller integer dtypes take fewer bytes per value.

### Practical Example: reading large files in chunks
```python
total = 0
for chunk in pd.read_csv('large_file.csv', chunksize=100000):
    total += chunk['revenue'].sum()
print(total)
```
This processes the file piece by piece instead of loading it entirely into memory.

### Common Mistakes
- Loading an entire multi-GB CSV with `pd.read_csv()` with no `usecols`/`dtype`/`chunksize`, causing memory errors.
- Using `apply()` row-by-row on millions of rows instead of vectorized NumPy/Pandas operations.
- Repeatedly calling `pd.concat()` inside a loop (quadratic cost) instead of collecting results in a list and concatenating once at the end.

### Exam Tips
- Key optimization keywords to mention: **dtype downcasting, category dtype, chunksize, vectorization, usecols**.

### Practice Question
Take the `demo_df` above and check the memory used by the `category` column alone before and after conversion using `memory_usage(deep=True)`.

<a id="s14"></a>
## 14. Complete Data-Cleaning Workflow (Case Study)

### Theory
A typical real-world cleaning pipeline follows these steps:
1. Load and inspect (`head`, `info`, `describe`)
2. Handle missing values
3. Handle duplicates
4. Fix data types (numeric/categorical/datetime)
5. Handle outliers / inconsistent categories
6. Feature engineering (new useful columns)
7. Final validation

We will apply this full pipeline to a fresh, deliberately messy dataset.

In [49]:
# Step 0: Create a deliberately messy dataset
raw = pd.DataFrame({
    'CustomerID': [1, 2, 2, 3, 4, 5, 6, 7, 8, 9],
    'Name': ['Aman', 'Priya', 'Priya', 'Rohit', 'sneha', 'Vikram', None, 'Aisha', 'Karan', 'Meera'],
    'Age': [25, 32, 32, np.nan, 41, 29, 35, 200, 22, 30],
    'City': ['Ludhiana', 'delhi', 'Delhi', 'Mumbai', 'Chennai', np.nan, 'Pune', 'Delhi', 'Ludhiana', 'Mumbai'],
    'PurchaseAmount': ['1200', '2500', '2500', '3200', '1800', '900', '4100', '2200', '1500', '2700']
})
raw


,CustomerID,Name,Age,City,PurchaseAmount
0,1,Aman,25.0,Ludhiana,1200
1,2,Priya,32.0,delhi,2500
2,2,Priya,32.0,Delhi,2500
3,3,Rohit,NaN,Mumbai,3200
4,4,sneha,41.0,Chennai,1800
5,5,Vikram,29.0,NaN,900
6,6,None,35.0,Pune,4100
7,7,Aisha,200.0,Delhi,2200
8,8,Karan,22.0,Ludhiana,1500
9,9,Meera,30.0,Mumbai,2700


In [50]:
# Step 1: Inspect
raw.info()
print("\nMissing values:\n", raw.isna().sum())
print("\nDuplicate rows:", raw.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      10 non-null     int64  
 1   Name            9 non-null      object 
 2   Age             9 non-null      float64
 3   City            9 non-null      object 
 4   PurchaseAmount  10 non-null     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 532.0+ bytes

Missing values:
 CustomerID        0
Name              1
Age               1
City              1
PurchaseAmount    0
dtype: int64

Duplicate rows: 0


In [51]:
# Step 2: Handle duplicates
clean = raw.drop_duplicates().copy()
print("Rows after removing duplicates:", len(clean))


Rows after removing duplicates: 10


In [52]:
# Step 3: Fix data types
clean['PurchaseAmount'] = clean['PurchaseAmount'].astype(float)
clean['Age'] = pd.to_numeric(clean['Age'], errors='coerce')
clean.dtypes


,0
CustomerID,int64
Name,object
Age,float64
City,object
PurchaseAmount,float64


In [53]:
# Step 4: Handle missing values
clean['Age'] = clean['Age'].fillna(clean['Age'].median())
clean['City'] = clean['City'].fillna('Unknown')
clean['Name'] = clean['Name'].fillna('Unknown')
clean.isna().sum()


,0
CustomerID,0
Name,0
Age,0
City,0
PurchaseAmount,0


In [54]:
# Step 5: Fix inconsistent categories (text case) and outliers
clean['City'] = clean['City'].str.strip().str.title()
clean['Name'] = clean['Name'].str.strip().str.title()

# Age of 200 is clearly an outlier/data-entry error -> cap using domain knowledge
clean.loc[clean['Age'] > 100, 'Age'] = clean['Age'].median()

clean[['Name', 'City', 'Age']]


,Name,City,Age
0,Aman,Ludhiana,25.0
1,Priya,Delhi,32.0
2,Priya,Delhi,32.0
3,Rohit,Mumbai,32.0
4,Sneha,Chennai,41.0
5,Vikram,Unknown,29.0
6,Unknown,Pune,35.0
7,Aisha,Delhi,32.0
8,Karan,Ludhiana,22.0
9,Meera,Mumbai,30.0


In [55]:
# Step 6: Feature engineering
clean['AgeGroup'] = pd.cut(clean['Age'], bins=[0, 25, 35, 50, 100],
                            labels=['Young', 'Adult', 'MiddleAge', 'Senior'])
clean['HighSpender'] = clean['PurchaseAmount'] > clean['PurchaseAmount'].mean()
clean


,CustomerID,Name,Age,City,PurchaseAmount,AgeGroup,HighSpender
0,1,Aman,25.0,Ludhiana,1200.0,Young,False
1,2,Priya,32.0,Delhi,2500.0,Adult,True
2,2,Priya,32.0,Delhi,2500.0,Adult,True
3,3,Rohit,32.0,Mumbai,3200.0,Adult,True
4,4,Sneha,41.0,Chennai,1800.0,MiddleAge,False
5,5,Vikram,29.0,Unknown,900.0,Adult,False
6,6,Unknown,35.0,Pune,4100.0,Adult,True
7,7,Aisha,32.0,Delhi,2200.0,Adult,False
8,8,Karan,22.0,Ludhiana,1500.0,Young,False
9,9,Meera,30.0,Mumbai,2700.0,Adult,True


In [56]:
# Step 7: Final validation
print("Final shape:", clean.shape)
print("Any missing values left?:", clean.isna().sum().sum() == 0)
print("Any duplicates left?:", clean.duplicated().sum() == 0)
clean.describe(include='all')


Final shape: (10, 7)
Any missing values left?: True
Any duplicates left?: False


,CustomerID,Name,Age,City,PurchaseAmount,AgeGroup,HighSpender
count,10.000000,10,10.00000,10,10.000000,10,10
unique,NaN,9,NaN,6,NaN,3,2
top,NaN,Priya,NaN,Delhi,NaN,Adult,False
freq,NaN,2,NaN,3,NaN,7,5
mean,4.700000,NaN,31.00000,NaN,2260.000000,NaN,NaN
std,2.750757,NaN,5.18545,NaN,962.866092,NaN,NaN
min,1.000000,NaN,22.00000,NaN,900.000000,NaN,NaN
25%,2.250000,NaN,29.25000,NaN,1575.000000,NaN,NaN
50%,4.500000,NaN,32.00000,NaN,2350.000000,NaN,NaN
75%,6.750000,NaN,32.00000,NaN,2650.000000,NaN,NaN


**Explanation:** Each step solved one specific real-world problem: duplicate customer `Priya`, inconsistent city casing (`delhi` vs `Delhi`), a text-stored numeric column (`PurchaseAmount`), a missing `Age`, an impossible `Age` of 200, and a missing `Name`/`City`. This is the standard structure expected in a "describe your data cleaning process" exam answer.

### Common Mistakes
- Cleaning steps done in the wrong order (e.g., computing statistics before removing duplicates, which biases the results).
- Not re-validating after cleaning (skipping Step 7) — bugs can silently remain.
- Deleting outliers without domain judgement — sometimes capping/imputing is better than dropping.

### Exam Tips
- Always state the cleaning pipeline as an ordered list — examiners award marks for structure, not just code.

### Practice Question
Take the `raw` dataset again and write the full cleaning pipeline yourself using different fill strategies (e.g., fill `Age` with mean instead of median), then compare the final result.

<a id="s15"></a>
## 15. Common Pandas Mistakes (Consolidated)

| Mistake | Why it's wrong | Correct Approach |
|---|---|---|
| `df[col1, col2]` | Wrong syntax; raises `KeyError` | `df[['col1', 'col2']]` |
| Using `and`/`or` for combining conditions | Python's `and`/`or` don't work element-wise on Series | Use `&`, `\|`, `~` with parentheses |
| Chained indexing: `df[df.x > 0]['y'] = 1` | May trigger `SettingWithCopyWarning`; may not modify the original | Use `df.loc[df.x > 0, 'y'] = 1` |
| Assuming methods modify in place | Most Pandas methods return a **new** object | Assign the result back, or pass `inplace=True` explicitly |
| Comparing floats with `==` for missing check | `NaN != NaN`, so `df['x'] == np.nan` never works | Use `df['x'].isna()` |
| Ignoring index alignment during arithmetic | Operations align by index; mismatched indices produce `NaN` | Use `reset_index()` or `.values` intentionally when needed |
| Using `apply()` for simple math | Slow on large data | Use vectorized operations |
| Not specifying `dtype`/`parse_dates` when reading CSVs | Leads to wrong types (e.g., dates read as text) | Use `dtype=` and `parse_dates=` in `read_csv` |
| Merging without checking for duplicate keys | Silently multiplies rows (fan-out) | Validate key uniqueness before merging, or use `validate=` in `merge` |
| Forgetting `observed=True` with categorical groupby (recent Pandas) | May produce unexpected empty groups or a FutureWarning | Pass `observed=True` explicitly |

### Exam Tips
- Viva examiners love asking "what is `SettingWithCopyWarning` and how do you avoid it?" — answer: it warns that you may be modifying a **copy** of the DataFrame instead of the original due to chained indexing; avoid it by using a single `.loc[]` call.

<a id="s16"></a>
## 16. Master Comparison Tables (Quick Reference)

### Series vs DataFrame
| Feature | Series | DataFrame |
|---|---|---|
| Dimensions | 1-D | 2-D |
| Structure | Single labeled column | Multiple columns |

### loc vs iloc
| | loc | iloc |
|---|---|---|
| Basis | Label | Position |
| Slice end | Inclusive | Exclusive |

### merge vs concat vs join
| | merge | concat | join |
|---|---|---|---|
| Combines by | Key column(s) | Axis/position | Index |
| SQL analogy | JOIN | UNION | JOIN (index) |

### groupby vs pivot_table
| | groupby | pivot_table |
|---|---|---|
| Output | Long format | Wide/grid format |
| Default aggregation | None (must specify) | mean |

### count vs size
| | count() | size() |
|---|---|---|
| Ignores NaN | Yes | No |

### mean vs median
| | mean | median |
|---|---|---|
| Outlier sensitivity | High | Low |

### inner / left / right / outer
| Join | Result |
|---|---|
| inner | Matching rows only |
| left | All left rows |
| right | All right rows |
| outer | All rows from both |

### axis=0 vs axis=1
| | axis=0 | axis=1 |
|---|---|---|
| Direction | Down rows | Across columns |

<a id="s17"></a>
## 17. Exam Revision and Cheat Sheet

### Creating Data
```python
pd.Series(data, index=...)
pd.DataFrame({'col': [...]})
pd.read_csv('file.csv')
```

### Inspecting Data
```python
df.head(); df.tail(); df.shape; df.info(); df.describe(); df.columns; df.dtypes
```

### Selecting Data
```python
df['col']                # Series
df[['col1','col2']]      # DataFrame
df.loc[row_label, col]   # label based
df.iloc[row_pos, col_pos]# position based
df[df['col'] > 5]        # boolean filtering
```

### Cleaning Data
```python
df.isna().sum()
df.dropna(); df.dropna(subset=['col'])
df.fillna(value); df['col'].fillna(df['col'].mean())
df.duplicated(); df.drop_duplicates()
df['col'].astype('category')
pd.to_datetime(df['col']); pd.to_numeric(df['col'], errors='coerce')
```

### Transforming Data
```python
df.rename(columns={'old':'new'})
df.sort_values('col', ascending=False)
df['col'].map({...})
df['col'].apply(lambda x: ...)
df.apply(lambda row: ..., axis=1)
np.where(condition, val_if_true, val_if_false)
```

### Aggregating Data
```python
df.groupby('col')['target'].sum()
df.groupby(['c1','c2']).agg({'target': ['sum','mean']})
pd.pivot_table(df, values='v', index='i', columns='c', aggfunc='sum')
pd.crosstab(df['c1'], df['c2'])
```

### Combining Data
```python
pd.concat([df1, df2], axis=0)   # stack rows
pd.concat([df1, df2], axis=1)   # stack columns
pd.merge(df1, df2, on='key', how='inner'/'left'/'right'/'outer')
df1.join(df2, how='left')
```

### Time Series
```python
df['date'].dt.year / .dt.month / .dt.day_name()
df.set_index('date').resample('M').sum()
df['col'].rolling(window=3).mean()
```

### Performance
```python
df.memory_usage(deep=True)
df['col'].astype('int32')  # downcast
pd.read_csv('file.csv', chunksize=100000, usecols=[...], dtype={...})
```

<a id="s18"></a>
## 18. Viva Questions (25, with Answers)

**1. What is Pandas?**
A Python library for fast, flexible data manipulation and analysis, built on top of NumPy, providing Series and DataFrame structures.

**2. What is the difference between a Series and a DataFrame?**
A Series is 1-dimensional (one labeled column); a DataFrame is 2-dimensional (multiple columns, each a Series).

**3. How do you check for missing values?**
`df.isna()` or `df.isnull()`, often combined with `.sum()` to count them per column.

**4. What is the difference between `dropna()` and `fillna()`?**
`dropna()` removes rows/columns containing missing values; `fillna()` replaces missing values with a specified value or strategy.

**5. What does `duplicated()` return?**
A boolean Series marking rows that are duplicates of an earlier row (True) or not (False).

**6. Difference between `loc` and `iloc`?**
`loc` selects by label; `iloc` selects by integer position. `loc` slicing is inclusive of the end label; `iloc` slicing is exclusive.

**7. What is the split-apply-combine strategy?**
The three-step process behind `groupby()`: split data into groups, apply a function to each group, then combine the results into one structure.

**8. Difference between `merge()` and `concat()`?**
`merge()` combines DataFrames using common key column(s), like a SQL join; `concat()` simply stacks DataFrames along an axis without matching keys.

**9. What are the four types of joins in `merge()`?**
`inner` (matching rows only), `left` (all left rows), `right` (all right rows), `outer` (all rows from both).

**10. What is the difference between `map()` and `apply()`?**
`map()` works only on a Series for element-wise substitution/transformation; `apply()` works on a Series or DataFrame and can perform more general row-wise/column-wise operations.

**11. Why is vectorization preferred over `apply()`/loops?**
Vectorized operations run in optimized, compiled C code under the hood and process entire arrays at once, making them significantly faster than Python-level iteration.

**12. What is a pivot table?**
A pivot table reshapes data into a grid, aggregating a numeric value across two categorical dimensions (rows and columns), similar to Excel pivot tables.

**13. Difference between `pivot_table()` and `crosstab()`?**
`pivot_table` aggregates a numeric column with a chosen function (mean by default); `crosstab` counts frequencies of category combinations by default (though it can also aggregate with `values`/`aggfunc`).

**14. What is the `category` dtype and why use it?**
A Pandas dtype for columns with a limited set of repeated values; it saves memory and speeds up group operations compared to plain text (`object`) columns.

**15. What does `resample()` do?**
Groups time-series data into fixed time intervals (e.g., daily, weekly, monthly) and applies an aggregation, similar to `groupby` but specifically for datetime indexes.

**16. What is a rolling window / moving average?**
A calculation (like mean) performed over a sliding window of consecutive rows, used to smooth out short-term fluctuations in time-series data.

**17. What is `SettingWithCopyWarning`?**
A warning Pandas raises when an operation may be performed on a copy of a DataFrame instead of the original, often caused by chained indexing (e.g., `df[cond]['col'] = x`). Avoided by using a single `.loc[]` assignment.

**18. Why doesn't `df['col'] == np.nan` work to detect missing values?**
Because `NaN` is not equal to anything, including itself, by IEEE floating-point rules; use `.isna()` instead.

**19. What does `axis=0` vs `axis=1` mean?**
`axis=0` refers to operating down the rows (per column); `axis=1` refers to operating across the columns (per row).

**20. What is the difference between `count()` and `size()` in `groupby`?**
`count()` counts non-null values per column within each group; `size()` counts all rows in each group regardless of missing values, and returns a single Series.

**21. How do you convert a column to datetime?**
`pd.to_datetime(df['col'])`, optionally with a `format=` argument for non-standard date strings.

**22. What is `inplace=True` and why should it be used carefully?**
It tells a Pandas method to modify the DataFrame directly instead of returning a new one; the method then returns `None`, so `df = df.method(inplace=True)` would incorrectly set `df` to `None`.

**23. How do you optimize memory for a large DataFrame?**
Downcast numeric dtypes (`int64`->`int32`/`int8`), convert repeated text columns to `category`, select only needed columns with `usecols`, and process large files in `chunksize` batches.

**24. What is the difference between `df.mean()` and `df['col'].mean()`?**
`df.mean()` computes the mean of every numeric column in the DataFrame (returns a Series); `df['col'].mean()` computes the mean of a single specified column (returns a scalar).

**25. What does `reset_index()` do and when is it needed?**
It converts the current index back into a regular column and creates a new default integer index; commonly needed after a `groupby()` aggregation to turn the grouped labels back into normal columns for further processing.

<a id="s19"></a>
## 19. Exam Question Bank

### A. Very Short Answer Questions (15)
1. Define a Pandas Series.
2. Define a Pandas DataFrame.
3. What function checks for missing values?
4. What does `drop_duplicates()` do?
5. Name the two Pandas indexers used for selection.
6. What does `groupby()` implement (name the strategy)?
7. What is the default aggregation function of `pivot_table()`?
8. Name any two join types supported by `merge()`.
9. Which accessor is used to extract date parts from a datetime column?
10. What does `rolling()` compute?
11. Which dtype saves memory for repeated text categories?
12. What is the output type of `df['col']`?
13. What is the output type of `df[['col']]`?
14. What keyword argument makes a Pandas method modify the original DataFrame?
15. Which function converts a string column to numeric, forcing errors to `NaN`?

### B. Short Answer Questions (15)
1. Explain the difference between `loc` and `iloc` with an example.
2. Explain the difference between `dropna()` and `fillna()`.
3. What is the difference between `df.isna()` and `df.notna()`?
4. Explain `value_counts()` with an example use case.
5. Explain the difference between `map()` and `apply()`.
6. What is vectorization and why is it faster than `apply()`?
7. Explain the purpose of `pd.crosstab()`.
8. Explain the four join types (`inner`, `left`, `right`, `outer`) briefly.
9. What is the difference between `concat()` and `merge()`?
10. Explain what `resample()` does with an example frequency code.
11. What is a rolling/moving average and why is it useful?
12. Explain how to detect and remove duplicate rows.
13. What is `SettingWithCopyWarning` and how do you avoid it?
14. Explain the difference between `count()` and `size()` in a groupby.
15. Explain how the `category` dtype helps optimize memory.

### C. Long Answer Questions (10)
1. Describe the complete data-cleaning workflow you would follow on a new, messy dataset, with at least six clearly labeled steps.
2. Explain `groupby()` in depth, covering the split-apply-combine model, with a worked example using multiple aggregation functions.
3. Compare `merge()`, `concat()`, and `join()` in detail, including syntax, use cases, and at least one example each.
4. Explain all four join types in `merge()` with example DataFrames and expected outputs.
5. Discuss strategies for optimizing Pandas performance on large datasets (at least four techniques), explaining why each helps.
6. Explain time-series functionality in Pandas: datetime conversion, the `.dt` accessor, resampling, and rolling windows, with examples.
7. Explain the difference between `apply()`, `map()`, and vectorized operations, including a performance comparison.
8. Discuss different strategies for handling missing data (deletion vs imputation) and when each is appropriate.
9. Explain pivot tables and crosstabs in depth, including when to prefer one over the other, with examples.
10. Discuss at least eight common Pandas mistakes beginners make and how to avoid each one.

### D. Practical / Coding Questions (15)
1. Create a DataFrame of 6 employees with `Name`, `Department`, and `Salary`, then print employees earning above the average salary.
2. Given a DataFrame with a `Date` column as strings, convert it to datetime and extract the month name into a new column.
3. Create a Series with at least two missing values and demonstrate `dropna()`, `fillna()` with mean, and `fillna()` with forward-fill.
4. Given a DataFrame with duplicate rows, show how to find and remove them, keeping only the last occurrence.
5. Write code to group a sales DataFrame by `Category` and compute total and average `Revenue` in a single `agg()` call.
6. Write a pivot table showing average `Marks` of students by `Subject` (rows) and `Class` (columns).
7. Create two small DataFrames sharing a `key` column and demonstrate `inner`, `left`, `right`, and `outer` merges.
8. Demonstrate the difference between `loc[2:5]` and `iloc[2:5]` on any DataFrame with a default integer index.
9. Write a lambda function using `apply()` to categorize ages into `'Minor'` (<18) and `'Adult'` (>=18).
10. Convert a `category`-eligible text column to `category` dtype and show the memory savings using `memory_usage(deep=True)`.
11. Create a time-indexed DataFrame and compute a 7-day rolling mean of a numeric column.
12. Write code to resample daily data into monthly totals.
13. Use `crosstab()` to find how many students passed/failed per class.
14. Demonstrate chained-indexing's `SettingWithCopyWarning` and show the corrected version using `.loc`.
15. Write a full mini data-cleaning script: load a messy DataFrame, remove duplicates, fix data types, handle missing values, and validate the result.

<a id="s20"></a>
## 20. Hands-On Exercises

Attempt each exercise yourself in the empty code cell provided before checking the Solutions section (Section 21). A fresh, deterministic dataset is created below for these exercises.

In [57]:
# Dataset for the hands-on exercises (deterministic)
rng2 = np.random.default_rng(7)

employees = pd.DataFrame({
    'EmpID': range(1, 16),
    'Name': [f'Emp_{i}' for i in range(1, 16)],
    'Department': rng2.choice(['Sales', 'IT', 'HR', 'Finance'], size=15),
    'Salary': rng2.integers(30000, 90000, size=15),
    'JoinDate': pd.date_range('2021-01-01', periods=15, freq='45D'),
    'Rating': rng2.choice([1, 2, 3, 4, 5, np.nan], size=15)
})
employees.loc[[1, 6], 'Department'] = np.nan
employees = pd.concat([employees, employees.iloc[[2]]], ignore_index=True)
employees


,EmpID,Name,Department,Salary,JoinDate,Rating
0,1,Emp_1,Finance,79273,2021-01-01,4.0
1,2,Emp_2,NaN,37886,2021-02-15,4.0
2,3,Emp_3,HR,77824,2021-04-01,4.0
3,4,Emp_4,Finance,37144,2021-05-16,NaN
4,5,Emp_5,HR,58076,2021-06-30,5.0
5,6,Emp_6,Finance,78987,2021-08-14,5.0
6,7,Emp_7,NaN,48181,2021-09-28,5.0
7,8,Emp_8,Sales,50496,2021-11-12,4.0
8,9,Emp_9,Sales,46705,2021-12-27,3.0
9,10,Emp_10,IT,73168,2022-02-10,NaN


**Exercise 1:** Display the shape, column data types, and count of missing values per column of `employees`.

**Exercise 2:** Remove duplicate rows from `employees`.

**Exercise 3:** Fill missing `Department` values with `'Unknown'` and missing `Rating` values with the column mean.

**Exercise 4:** Using `loc`, select all employees in the `'IT'` department earning more than 50000.

**Exercise 5:** Add a new column `SalaryBand` using `apply()`/lambda: `'Low'` if Salary < 50000, `'Medium'` if 50000-75000, else `'High'`.

**Exercise 6:** Group by `Department` and compute the mean `Salary` and mean `Rating` in a single `agg()` call.

**Exercise 7:** Create a pivot table showing average `Salary` by `Department` (rows) and `SalaryBand` (columns).

**Exercise 8:** Extract the `JoinYear` and `JoinMonth` from `JoinDate` using the `.dt` accessor.

**Exercise 9:** Create a second small DataFrame mapping each `Department` to a `Location`, then left-merge it onto `employees`.

**Exercise 10:** Convert `Department` to `category` dtype and compare memory usage before and after using `memory_usage(deep=True)`.

In [58]:
# Use this cell to attempt the exercises yourself before checking Section 21.



<a id="s21"></a>
## 21. Solutions to Exercises

Solutions are shown below in order. Try the exercises yourself first for maximum benefit.

In [59]:
# Solution 1
print("Shape:", employees.shape)
print("\nDtypes:\n", employees.dtypes)
print("\nMissing values:\n", employees.isna().sum())


Shape: (16, 6)

Dtypes:
 EmpID                  int64
Name                  object
Department            object
Salary                 int64
JoinDate      datetime64[ns]
Rating               float64
dtype: object

Missing values:
 EmpID         0
Name          0
Department    2
Salary        0
JoinDate      0
Rating        4
dtype: int64


In [60]:
# Solution 2
employees_dedup = employees.drop_duplicates()
print("Rows before:", len(employees), "Rows after:", len(employees_dedup))
employees_dedup


Rows before: 16 Rows after: 15


,EmpID,Name,Department,Salary,JoinDate,Rating
0,1,Emp_1,Finance,79273,2021-01-01,4.0
1,2,Emp_2,NaN,37886,2021-02-15,4.0
2,3,Emp_3,HR,77824,2021-04-01,4.0
3,4,Emp_4,Finance,37144,2021-05-16,NaN
4,5,Emp_5,HR,58076,2021-06-30,5.0
5,6,Emp_6,Finance,78987,2021-08-14,5.0
6,7,Emp_7,NaN,48181,2021-09-28,5.0
7,8,Emp_8,Sales,50496,2021-11-12,4.0
8,9,Emp_9,Sales,46705,2021-12-27,3.0
9,10,Emp_10,IT,73168,2022-02-10,NaN


In [61]:
# Solution 3
employees_clean = employees_dedup.copy()
employees_clean['Department'] = employees_clean['Department'].fillna('Unknown')
employees_clean['Rating'] = employees_clean['Rating'].fillna(employees_clean['Rating'].mean())
employees_clean


,EmpID,Name,Department,Salary,JoinDate,Rating
0,1,Emp_1,Finance,79273,2021-01-01,4.000000
1,2,Emp_2,Unknown,37886,2021-02-15,4.000000
2,3,Emp_3,HR,77824,2021-04-01,4.000000
3,4,Emp_4,Finance,37144,2021-05-16,3.636364
4,5,Emp_5,HR,58076,2021-06-30,5.000000
5,6,Emp_6,Finance,78987,2021-08-14,5.000000
6,7,Emp_7,Unknown,48181,2021-09-28,5.000000
7,8,Emp_8,Sales,50496,2021-11-12,4.000000
8,9,Emp_9,Sales,46705,2021-12-27,3.000000
9,10,Emp_10,IT,73168,2022-02-10,3.636364


In [62]:
# Solution 4
it_high_earners = employees_clean.loc[
    (employees_clean['Department'] == 'IT') & (employees_clean['Salary'] > 50000)
]
it_high_earners


,EmpID,Name,Department,Salary,JoinDate,Rating
9,10,Emp_10,IT,73168,2022-02-10,3.636364
14,15,Emp_15,IT,60272,2022-09-23,3.636364


In [63]:
# Solution 5
def salary_band(s):
    if s < 50000:
        return 'Low'
    elif s <= 75000:
        return 'Medium'
    else:
        return 'High'

employees_clean['SalaryBand'] = employees_clean['Salary'].apply(salary_band)
employees_clean[['Name', 'Salary', 'SalaryBand']]


,Name,Salary,SalaryBand
0,Emp_1,79273,High
1,Emp_2,37886,Low
2,Emp_3,77824,High
3,Emp_4,37144,Low
4,Emp_5,58076,Medium
5,Emp_6,78987,High
6,Emp_7,48181,Low
7,Emp_8,50496,Medium
8,Emp_9,46705,Low
9,Emp_10,73168,Medium


In [64]:
# Solution 6
dept_summary = employees_clean.groupby('Department', observed=True).agg({
    'Salary': 'mean',
    'Rating': 'mean'
})
dept_summary


,Salary,Rating
Department,,
Finance,68307.000000,3.654545
HR,67950.000000,4.500000
IT,59577.333333,3.424242
Sales,51963.000000,2.666667
Unknown,43033.500000,4.500000


In [65]:
# Solution 7
salary_pivot = pd.pivot_table(
    employees_clean, values='Salary', index='Department', columns='SalaryBand',
    aggfunc='mean', fill_value=0, observed=True
)
salary_pivot


SalaryBand,High,Low,Medium
Department,,,
Finance,82562.333333,37144.0,56704.0
HR,77824.000000,0.0,58076.0
IT,0.000000,45292.0,66720.0
Sales,0.000000,46705.0,54592.0
Unknown,0.000000,43033.5,0.0


In [66]:
# Solution 8
employees_clean['JoinYear'] = employees_clean['JoinDate'].dt.year
employees_clean['JoinMonth'] = employees_clean['JoinDate'].dt.month
employees_clean[['JoinDate', 'JoinYear', 'JoinMonth']]


,JoinDate,JoinYear,JoinMonth
0,2021-01-01,2021,1
1,2021-02-15,2021,2
2,2021-04-01,2021,4
3,2021-05-16,2021,5
4,2021-06-30,2021,6
5,2021-08-14,2021,8
6,2021-09-28,2021,9
7,2021-11-12,2021,11
8,2021-12-27,2021,12
9,2022-02-10,2022,2


In [67]:
# Solution 9
dept_location = pd.DataFrame({
    'Department': ['Sales', 'IT', 'HR', 'Finance', 'Unknown'],
    'Location': ['Delhi', 'Bangalore', 'Mumbai', 'Chennai', 'NotAssigned']
})
employees_merged = pd.merge(employees_clean, dept_location, on='Department', how='left')
employees_merged[['Name', 'Department', 'Location']]


,Name,Department,Location
0,Emp_1,Finance,Chennai
1,Emp_2,Unknown,NotAssigned
2,Emp_3,HR,Mumbai
3,Emp_4,Finance,Chennai
4,Emp_5,HR,Mumbai
5,Emp_6,Finance,Chennai
6,Emp_7,Unknown,NotAssigned
7,Emp_8,Sales,Delhi
8,Emp_9,Sales,Delhi
9,Emp_10,IT,Bangalore


In [68]:
# Solution 10
before_mem = employees_clean['Department'].memory_usage(deep=True)
employees_clean['Department'] = employees_clean['Department'].astype('category')
after_mem = employees_clean['Department'].memory_usage(deep=True)
print(f"Before: {before_mem} bytes")
print(f"After : {after_mem} bytes")
print(f"Reduction: {(1 - after_mem/before_mem)*100:.1f}%")


Before: 929 bytes
After : 575 bytes
Reduction: 38.1%


---
## End of Notebook

This notebook covered the complete Module 2 syllabus: Series/DataFrame fundamentals, indexing, wrangling, categorical/numerical data, missing and duplicate values, aggregation and GroupBy, apply/map/lambda/vectorization, pivot tables and crosstab, merge/join/concat, time-series analysis, large-dataset optimization, a full cleaning workflow, common mistakes, all required comparison tables, a cheat sheet, 25 viva questions, a full exam question bank, and 10 solved hands-on exercises.

Revise Section 17 (Cheat Sheet) and Section 18 (Viva Questions) the night before an exam for a fast recap.